## README:

* Cells labeled with '## ----- CONFIG ----- ##' contain parameters that need to be set manually

In [1]:
# NOTE: UNCOMMENT ALL LINES AND RUN CELL IF RUNNING NOTEBOOK ON GOOGLE COLAB
!pip install dotenv
!pip install FinRL
!pip install stable_baselines3 sb3_contrib --upgrade
!pip install alpaca_trade_api
!pip install exchange_calendars
!pip install stockstats
!pip install wrds
!pip install websockets
!pip install yfinance
!pip install ta

  Using cached websockets-10.4-cp312-cp312-linux_x86_64.whl
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires websockets>=14.0, but you have websockets 10.4 which is incompatible.
google-adk 1.14.1 requires PyYAML<7.0.0,>=6.0.2, but you have pyyaml 6.0.1 which is incompatible.
google-adk 1.14.1 requires websockets<16.0.0,>=15.0.1, but you have websockets 10.4 which is incompatible.
google-genai 1.41.0 requires websockets<15.1.0,>=13.0.0, but you have websockets 10.4 which is incompatible.
gradio-client 1.13.3 requires websockets<16.0,>=13.0, but you have websockets 10.4 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets

In [2]:
import os
import sys

# add project root to path
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_path)

In [ ]:
# NOTE: UNCOMMENT ALL LINES AND RUN CELL IF RUNNING NOTEBOOK ON GOOGLE COLAB
import os
import sys
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# NOTE: CHANGE PATH AS NECESSARY
PATH = '/content/drive/MyDrive/ELEN6885 Final Project'
sys.path.append(os.path.join(PATH, 'rl-trading'))
os.makedirs(os.path.join(PATH, 'data'), exist_ok=True)
os.makedirs(os.path.join(PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(PATH, 'tensorboard_logs'), exist_ok=True)

Mounted at /content/drive


In [ ]:
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from gymnasium import spaces
from finrl.meta.preprocessor.preprocessors import data_split
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

from trading_environment.custom_env import StockTradingEnv
from trading_environment import rewards
from experiments.normalization import MaskedVecNormalize
from experiments.early_stopping import MultiCondEarlyStop
from evaluation.backtest import run_backtest
from evaluation.visualize import plot_trades_on_price_from_backtest
from evaluation import financial_metrics

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
if not torch.cuda.is_available():
  raise Exception('Please use a GPU compute')

## Load env variables

In [ ]:
load_dotenv(r'../.env')

try:
  DATA_DIR = os.getenv('DATA_DIR', os.path.join(PATH, 'data'))
except:
  DATA_DIR = None
print(DATA_DIR)

In [ ]:
# record experiment start dt
import datetime

EXP_START_DT = datetime.datetime.now().strftime('%Y%m%d_%H%M')
EXP_START_DT

## MLP feature extractor

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(
        self, input_dim: int, hidden_dim: int = 64, output_dim: int = 1
    ):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: [batch_size, input_dim]
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = self.out(x)
        return x

In [ ]:
class MLPFeaturesExtractor(BaseFeaturesExtractor):
    """
    Wrap SimpleMLP as a feature extractor.
    """
    def __init__(
        self, observation_space: spaces.Box, features_dim: int = 64
    ):
        super().__init__(observation_space, features_dim)

        # shape: (window_size, feature_dim)
        # will only use the latest (current) state, ignore lagged states
        input_dim = observation_space.shape[1]

        self.mlp = SimpleMLP(
            input_dim=input_dim,
            hidden_dim=features_dim,
            output_dim=features_dim
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        # observations: [batch_size, 100, 71]
        # batch_size = n_envs in VecEnv
        x = observations[:, -1, :] # get current state
        return self.mlp(x)

## Train-test split and normalization

In [ ]:
## ----- CONFIG ----- ##
SPLIT_DATE = '2018-01-01' # format %Y-%m-%d
DATA_FILE_NAME = 'data_with_features.csv'

base_columns = [
    'date',	'tic', 'close',	'high',	'low', 'open', 'volume'
]

stock_features = [
    'atr_30',
    'macd',
    'macd_hist',
    'rsi_30'
]

economy_features = [
    'CPIAUCSL',
    'DFF',
    'ICSA',
    'T10Y2Y',
    'USEPUINDXD',
    'VIXCLS',
    # 'doy_sin',
    # 'doy_cos',
    'dow_sin',
    'dow_cos'
]

In [ ]:
# load finalized data
data_df = pd.read_csv(os.path.join(DATA_DIR, DATA_FILE_NAME))

data_df = data_df[base_columns + stock_features + economy_features]
stock_names = data_df['tic'].unique()
data_df['date'] = pd.to_datetime(data_df['date'], format='%Y-%m-%d')
data_df.shape

In [ ]:
SPLIT_DATE = pd.to_datetime(SPLIT_DATE, format='%Y-%m-%d')

# train-test split
min_date = data_df['date'].min()
max_date = data_df['date'].max()
print(min_date, max_date)

train_df = data_split(data_df, start=min_date, end=SPLIT_DATE)
trade_df = data_split(data_df, start=SPLIT_DATE, end=max_date + pd.Timedelta(days=1))

print(train_df.shape, trade_df.shape)
print(train_df['date'].min(), train_df['date'].max())
print(trade_df['date'].min(), trade_df['date'].max())
print('Any missing values in training:', train_df.isna().any().any())
print('Any missing values in testing:', trade_df.isna().any().any())

## Environment configuration

In [ ]:
## ----- CONFIG ----- ##
price = 'close'         # name of price to use for transactions / rewards
hmax = 1000             # maximum number of stocks to transact at each step
initial_amount = 5000   # initial amount of cash
buy_cost_pct = 0.005    # % cost of each stock purchase
sell_cost_pct = 0.005   # % cost of each stock sale
num_stock_shares = 0    # number of starting shares
window_size = 50        # number of previous time steps to encode
# reward_func = rewards.PnLReward()
reward_func = rewards.PenalizedTurnover(penalty=5e-3)
reward_scaling = 1      # reward scaling

In [ ]:
# environment kwargs
stock_dim = len(data_df['tic'].unique())
state_space = 1 + 2 * stock_dim + len(stock_features) * stock_dim + len(economy_features)

env_kwargs = {
    "price": price,
    "hmax": hmax,
    "initial_amount": initial_amount,
    "buy_cost_pct": [buy_cost_pct] * stock_dim,
    "sell_cost_pct": [sell_cost_pct] * stock_dim,
    "stock_dim": stock_dim,
    "state_space": state_space,
    "action_space": stock_dim,
    "reward_function": reward_func,
    "reward_scaling": 1,
    "num_stock_shares": [num_stock_shares] * stock_dim,
    "price": "close",
    "stock_features": stock_features,
    "economy_features": economy_features,
    "window_size": window_size
}


## Data normalization

In [ ]:
## ----- CONFIG ----- ##
PARALLEL_RUNS = 10

In [ ]:
# do not normalize prices, since they are used to compute rewards
# feature index starts after cash + prices * stock_dim + shares * stock_dim
feature_start_idx = 1 + 2 * stock_dim

In [ ]:
def make_train_env():
    env = StockTradingEnv(df=train_df, **env_kwargs)
    env = Monitor(env, filename=f"training_monitor_{EXP_START_DT}", allow_early_resets=True)
    return env

def make_valid_env():
    env = StockTradingEnv(df=trade_df, **env_kwargs)
    env = Monitor(env, filename=f"trading_monitor_{EXP_START_DT}", allow_early_resets=True)
    return env

In [ ]:
train_vec_env = DummyVecEnv([make_train_env for _ in range(PARALLEL_RUNS)])

obs_shape = train_vec_env.observation_space.shape   # (window_size, feature_dim)

# features_to_normalize = list of indices along the *feature_dim* axis
normalize_mask = np.zeros(obs_shape, dtype=bool)
# broadcast those feature indices over all time steps
normalize_mask[:, feature_start_idx:] = True

# Wrap training env with MaskedVecNormalize (this FITS the stats)
train_norm_env = MaskedVecNormalize(
    train_vec_env,
    normalize_mask=normalize_mask.any(axis=0),
    norm_obs=True,
    norm_reward=False,     # trading: keep reward raw
    clip_obs=10.0,
    training=True
)

## Model training

In [ ]:
## ----- CONFIG ----- ##
policy_kwargs = dict(
    net_arch=[64, 64],
    lstm_hidden_size=32,
    n_lstm_layers=3,
    shared_lstm=False,
    enable_critic_lstm=True,
    features_extractor_class=MLPFeaturesExtractor,
    features_extractor_kwargs=dict(features_dim=64),
)

model = RecurrentPPO(
    policy="MlpLstmPolicy",
    env=train_norm_env,

    # --- PPO rollout & update ---
    n_steps=64,
    batch_size=32,
    n_epochs=5,
    gamma=0.99,
    gae_lambda=0.95,

    # --- Optimization ---
    learning_rate=3e-4,
    max_grad_norm=0.5,
    clip_range=0.15,
    vf_coef=0.5,
    ent_coef=0.10,

    # --- Misc ---
    verbose=0,
    tensorboard_log=os.path.join(PATH, 'tensorboard_logs'),
    policy_kwargs=policy_kwargs,
)

In [ ]:
total_params = sum(p.numel() for p in model.policy.parameters())
trainable_params = sum(p.numel() for p in model.policy.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


In [ ]:
# early-stopping constraints are non-binding
# used to prevent over-usage of compute
# learning will complete in 1M steps
callback = MultiCondEarlyStop(
    loss_threshold=-1e6,
    loss_window=4,
    max_time_seconds=3600*6,
    verbose=0,
)

log_name = f"mlp_recur_ppo_{EXP_START_DT}"
model.learn(
    total_timesteps=1_000_000,
    callback=callback,
    log_interval=5,
    tb_log_name=log_name
)

### Save Model to Google Drive

In [ ]:
model_save_path = os.path.join(PATH, 'models', f'RecurrentPPO_{EXP_START_DT}.zip')
model.save(model_save_path)
print(f"Model saved successfully to: {model_save_path}")

## Check training logs

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./mlp_recur_ppo_tb

## Quick evaluation

In [ ]:
valid_vec_env = DummyVecEnv([make_valid_env])

# New MaskedVecNormalize for validation, SAME MASK, but NO updating
valid_norm_env = MaskedVecNormalize(
    valid_vec_env,
    normalize_mask=normalize_mask.any(axis=0),
    norm_obs=True,
    norm_reward=False,
    clip_obs=10.0,
    training=False
)

# Copy fitted stats from training to validation
valid_norm_env.obs_rms = train_norm_env.obs_rms
valid_norm_env.training = False
valid_norm_env.norm_reward = False

In [ ]:
# do not randomize starting_step
valid_env_kwargs = {**env_kwargs, 'starting_step': 0}

res = run_backtest(
    model,
    valid_norm_env,
    valid_env_kwargs
)
res.shape

In [ ]:
res.to_csv(os.path.join(PATH, 'data', f'backtesting_base_results_{EXP_START_DT}.csv'), index=False)

In [ ]:
financial_metrics.net_asset_change(res)

In [ ]:
%matplotlib inline
tickers = trade_df['tic'].unique()

for tic in tickers:
    fig, ax = plot_trades_on_price_from_backtest(res, ticker=tic)

## Format tensorboard logs

In [ ]:
# IN-CODE CITATION:
# Generated by ChatGPT 5.2 (OpenAI) on Dec 14, 2025.
# Prompt used:
# """Give me a Python function to read the tensorboard log generated by
# SB3 model.learn(tb_log_name='...') and write the logs as multiple .csv files
# to the same log folder that contains the events file.
# """
# Code has been manually tested and verified for accuracy and slightly modified.
import os, glob
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def tb_run_to_csvs(run_dir: str, out_dir: str | None = None) -> str:
    """
    Converts TensorBoard event scalars in `run_dir` into one CSV per scalar tag.
    Returns the output directory path.
    """
    run_dir = os.path.expanduser(run_dir)
    if out_dir is None:
        out_dir = os.path.join(run_dir, "csv_export")
    os.makedirs(out_dir, exist_ok=True)

    event_files = glob.glob(os.path.join(run_dir, "events.out.tfevents.*"))
    if not event_files:
        raise FileNotFoundError(f"No events.out.tfevents.* found in: {run_dir}")

    # Read all event files (some runs roll over into multiple files)
    all_tags = set()
    accumulators = []
    for ef in sorted(event_files, key=os.path.getmtime):
        ea = EventAccumulator(ef, size_guidance={"scalars": 0})
        ea.Reload()
        accumulators.append(ea)
        all_tags.update(ea.Tags().get("scalars", []))

    for tag in sorted(all_tags):
        rows = []
        for ea in accumulators:
            if tag in ea.Tags().get("scalars", []):
                rows.extend(ea.Scalars(tag))

        if not rows:
            continue

        df = pd.DataFrame({
            "wall_time": [r.wall_time for r in rows],
            "step":      [r.step for r in rows],
            "value":     [r.value for r in rows],
        }).sort_values("step")

        df.to_csv(os.path.join(out_dir, tag.replace("/", "__") + ".csv"), index=False)

    return out_dir

In [ ]:
run_dir = os.path.join(PATH, 'tensorboard_logs', f'{log_name}_1')
tb_run_to_csvs(run_dir)